In [ ]:
import numpy as np
import string

np.random.seed(42)

# ================================
#  PREPROCESS
# ================================
def remove_punctuation(s):
    return s.translate(str.maketrans('', '', string.punctuation))


def add2dict(d, k, v):
    if k not in d:
        d[k] = []
    d[k].append(v)


# ================================
#  BUILD MARKOV MODEL (2nd ORDER)
# ================================

initial = {}
first_order = {}
second_order = {}

for line in open('poems.txt'):
    tokens = remove_punctuation(line.lower().strip()).split()
    T = len(tokens)

    for i in range(T):
        word = tokens[i]

        if i == 0:
            initial[word] = initial.get(word, 0) + 1

        else:
            prev1 = tokens[i-1]

            if i == 1:
                add2dict(first_order, prev1, word)

            else:
                prev2 = tokens[i-2]
                add2dict(second_order, (prev2, prev1), word)

            if i == T - 1:
                add2dict(second_order, (prev1, word), 'END')


# ================================
#  NORMALIZE TO PROBABILITY
# ================================

# initial
total = sum(initial.values())
for k in initial:
    initial[k] /= total


def list2prob(d):
    out = {}
    total = len(d)

    for word in d:
        out[word] = out.get(word, 0) + 1

    for word in out:
        out[word] /= total

    return out


for k in first_order:
    first_order[k] = list2prob(first_order[k])

for k in second_order:
    second_order[k] = list2prob(second_order[k])


# ================================
#  SAMPLING FUNCTION
# ================================
def sample_word(prob_dict):
    r = np.random.random()
    cumulative = 0

    for word, prob in prob_dict.items():
        cumulative += prob
        if r < cumulative:
            return word

    return list(prob_dict.keys())[-1]


# ================================
#  GENERATE POEM
# ================================
def generate_poem(n_lines=6):

    for _ in range(n_lines):

        sentence = []

        # first word
        w0 = sample_word(initial)
        sentence.append(w0)

        # second word
        w1 = sample_word(first_order[w0])
        sentence.append(w1)

        # generate rest
        while True:
            key = (w0, w1)

            if key not in second_order:
                break

            w2 = sample_word(second_order[key])

            if w2 == 'END':
                break

            sentence.append(w2)
            w0, w1 = w1, w2

        print(" ".join(sentence))


# ================================
# RUN
# ================================
generate_poem(8)

that broods and sleeps on his soul
its time i took my mind
the advantages it has to wear
such as even poets would admit perforce
she was shut in for furniture
and now the other heaped along the sky
a fountain at my feet
earth fills her lap with pleasures of my heart
